In [ ]:
# %cd /content/drive/MyDrive/Github/BA/"Deep Knowledge Space Theory"
# %ls

##### Imports

In [56]:
import os
import numpy as np
import pandas as pd
import math
import random
import matplotlib.pyplot as plt

from tqdm import tqdm
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import _LRScheduler
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset

### Settings

In [57]:
# confere <Brinkmann, G. and McKay, B.D., 2002. Posets on up to 16 points. Order, 19, pp.147-179.>
num_posets = {2: 3, 3: 19, 4: 219, 5: 4231, 6: 130023, 7: 6129859}
num_states = {2: 4, 3: 8, 4: 16, 5: 32, 6: 64, 7: 128, 8: 256, 9: 512, 10: 1024}

# Hyperparameters
N = 5
BATCH_SIZE = 4
N_HIDDEN = 256 * 4

PAD_TOKEN = "<pad>"
EOS_TOKEN = "<eos>"
alphabet = "abcdefghijklmnopqrstuvwxyz"

### Model

In [58]:
class SimpleDecoderModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, vocab_size, n_heads=2, n_layers=2, dropout_value=0.1, max_seq_len=2**(N+1)):
        super(SimpleDecoderModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.number_heads = n_heads
        self.n_layers = n_layers
        self.dropout = dropout_value
        self.intermediate_dim = int(hidden_dim/2)

        # First linear layer
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        # Non-linearity
        # self.activation = nn.Tanh()
        # Second linear layer
        # self.intermediate_proj = nn.Linear(hidden_dim, hidden_dim)

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_encoder = PositionalEncoding(hidden_dim, max_seq_len, dropout_rate=self.dropout)
        self.decoder_layer = nn.TransformerDecoderLayer(d_model=hidden_dim, nhead=self.number_heads, activation="relu")
        self.transformer_decoder = nn.TransformerDecoder(self.decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(hidden_dim, vocab_size - 1) # exclude padding token from generation

    def forward(self, input_matrix, tgt_seq, precomputed_K_embedding=None):
        batch_size = input_matrix.shape[0]
        seq_length = tgt_seq.shape[1]

        tgt_emb = self.embedding(tgt_seq)
        tgt_emb = self.pos_encoder(tgt_emb)
        tgt_emb = tgt_emb.permute(1, 0, 2).float()
        
        if precomputed_K_embedding is None:
            K_embedding = self.input_proj(input_matrix.float())
        else:
            K_embedding = precomputed_K_embedding

        # memory = self.activation(memory)
        # memory = self.intermediate_proj(memory)
        memory = K_embedding.view(1, batch_size, self.hidden_dim)
        memory = memory.repeat(seq_length, 1, 1).float()

        output = self.transformer_decoder(tgt=tgt_emb, memory=memory)
        output = self.output_proj(output)

        return output, K_embedding

class PositionalEncoding(nn.Module):
    def __init__(self, hidden_dim, max_len=5000, dropout_rate=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout_rate)

        pe = torch.zeros(max_len, hidden_dim)
        position = torch.arange(0, max_len, dtype=torch.long).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2).long() * (-math.log(10000.0) / hidden_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        x = self.dropout(x)
        return x

def custom_CE_loss(output, labels):
    pad_token_id = output.shape[-1] # excluded pad tokens from generation
    output_flat = output.permute(1,2,0)
    mask = (labels != pad_token_id)

    loss = F.cross_entropy(output_flat, labels, ignore_index=pad_token_id, reduction='none')
    masked_loss = loss * mask

    averaged_loss = masked_loss.sum() / mask.long().sum()
    return averaged_loss


### Dataset

In [ ]:
# Data path: <data/sequences-Q{N}.json> for N > 6

In [59]:
# Parts of this section are based on lecture materials from <"Deep learning for NLP" (University of Osnabrueck, autumn 2023)>

def pad_sequences(sequences, pad_token=PAD_TOKEN):
    max_length = max(len(seq) for seq in sequences)
    padded_sequences = [
        seq + list((pad_token,) * (max_length - len(seq)))
        for seq in sequences
    ]
    return padded_sequences

# pad 2D matrixes to N x N, such that [[1,2],[3,4]] -> [[1,2,0,0], [3,4,0,0], [0,0,0,0], [0,0,0,0]] for N=4
def pad_matrices(matrices): 
    N = max(len(matrix) for matrix in matrices)

    padded_matrices = []

    for matrix in matrices:
        padded_matrix = []
        for i in range(N):
            if i < len(matrix):
                row = matrix[i] + [0]*(N-len(matrix[i]))
            else:
                row = [0]*N
            padded_matrix.append(row)
        padded_matrices.append(padded_matrix)

    return padded_matrices

def binary_to_alphabetic(seq):
    # transforms states from binary to alphabetic set representation:
    # ["00", "10", "11"] -> ["", "a", "ab"]
    # ["000", "001", "100", "101", "111"] -> ["", "c", "ac", "abc"]
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    alphabet = [c for c in alphabet]
    result = []
    for state in seq:
        new_state = ""
        for idx, item in enumerate(state):
            if item == "1":
                new_state += alphabet[idx]
        result.append(new_state)
    return result

def load_sequences(domain_sizes = [4]):
    all_sequences = []
    all_matrices = []

    for n in domain_sizes:
        path = f"data/sequences-Q{n}.json"
        seq_df = pd.read_json(path, orient="records")
        all_sequences += [list(seq) for seq in seq_df["Sequence"]]
        all_matrices += [list(m) for m in seq_df["Relations"]]

    sequences_transformed = [binary_to_alphabetic(seq) for seq in all_sequences]
    sequences_transformed = [seq + [EOS_TOKEN] for seq in sequences_transformed] # add eos_toeken to each sequence
    padded_sequences = pad_sequences(sequences_transformed)
    padded_matrices = pad_matrices(all_matrices)

    return padded_sequences, padded_matrices

def set_of_states_in(sequences):
    return {sublst for lst in sequences for sublst in lst}

# def sort_sequences(set_of_states):
#     array = np.array([[int(x) for x in sublist] for sublist in set_of_states])

#     def binary_to_int(binary_list):
#         return int("".join(str(x) for x in binary_list), 2)
#     sorted_array =  np.array(sorted(array, key=lambda sublist: (sum(sublist), -binary_to_int(sublist))))
#     sorted_strings = ["".join(str(x) for x in sublist) for sublist in sorted_array]

#     return sorted_strings

def sort_sequences_alphabetic(set_of_states):
    # sorts list of strings a) by length b) alphabetically like ""<"a"<"b"<"c"<"ab"<"ac"<"bc"<"abc"<"<pad>"
    sorted_states = sorted(set_of_states, key=lambda x: (len(x), x))
    return sorted_states

def create_state2idx(sequences):
    set_of_states = set_of_states_in(sequences)
    if PAD_TOKEN in set_of_states:
        set_of_states.remove(PAD_TOKEN)

    sorted_set_of_states = sort_sequences_alphabetic(list(set_of_states))
    state2idx = {state: idx for idx, state in enumerate(sorted_set_of_states)}

    state2idx[PAD_TOKEN] = len(state2idx)
    return state2idx

def encode_sequence(sequence, state2idx):
    return [state2idx[state] for state in sequence]

def encode_sequences(sequences, state2idx):
    return np.array([
        encode_sequence(sequence, state2idx)
        for sequence in tqdm(sequences)
    ])

class Sequences(Dataset):
    def __init__(self, domain_sizes=[2,3,4]):
        self.sequences, relations = load_sequences(domain_sizes=domain_sizes)
        self.state2idx = create_state2idx(self.sequences)
        self.vocab_size = len(self.state2idx.values())
        if self.state2idx is not None:
          print("Initialized properly.")
        else:
          print("Not initialized properly.")

        self.idx2state = {idx: state for state, idx in self.state2idx.items()}
        self.encoded = encode_sequences(self.sequences, self.state2idx)
        self.matrices_flat = np.array([np.array(r).flatten() for r in relations])

    def __getitem__(self, i):
        input_seq, target_seq = torch.tensor(self.encoded[i, :-1]), torch.tensor(self.encoded[i, 1:])
        matrix_repeated_tensor = torch.tensor(self.matrices_flat[i,:], dtype=torch.long)

        return input_seq, target_seq, matrix_repeated_tensor

    def __len__(self):
        return len(self.encoded)

In [60]:
# Test the dataset
dataset = Sequences(domain_sizes=[2,3,4])
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE)

# Display sample and target
train_features, train_labels, train_matrices = next(iter(train_loader))
print("Training data from Dataloader:")
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
print(f"Matrix batch shape: {train_matrices.size()}")

Initialized properly.


100%|██████████| 241/241 [00:00<?, ?it/s]

Training data from Dataloader:
Feature batch shape: torch.Size([4, 16])
Labels batch shape: torch.Size([4, 16])
Matrix batch shape: torch.Size([4, 16])


### Training

In [61]:
def train(model, train_loader, optimizer, criterion1, criterion2, scheduler, clip_norm, dataset, length_wheight, device):
    model.train()
    
    total_loss_CE = 0.
    total_loss_L = 0.
    total_loss_combiened = 0.
    norm = clip_norm

    for data, targets, relation in train_loader:
        data, targets, relation = data.to(device), targets.long().to(device), relation.to(device)
        optimizer.zero_grad()
        output, K_embedding_batch = model.forward(relation, data)
        
        # Cross-entropy loss
        loss = criterion1(output, targets)
        total_loss_CE += loss.item() * (1 - length_wheight) # * len(data) # Scale loss by batch size

        # Length loss: Allignmenzt between hidden representation's norms and sequence lengths (MSE loss with tolerance window)
        seq_lengths = ((targets != dataset.state2idx[PAD_TOKEN]) & (targets != dataset.state2idx[EOS_TOKEN])).sum(dim=1).float()
        embedding_norms = K_embedding_batch.norm(dim=1)
        length_loss = criterion2(embedding_norms, seq_lengths)
        total_loss_L += length_loss.item() * length_wheight # * len(data) # Scale loss by batch size
    
        loss_combined = (1-length_wheight) * loss + length_loss * length_wheight
        loss_combined.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), norm)
        
        total_loss_combiened += loss_combined
        optimizer.step()

    return total_loss_CE / len(train_loader), total_loss_L / len(train_loader), total_loss_combiened / len(train_loader)

def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.

    with torch.no_grad():
        for data, targets, relation in data_loader:
            data, targets, relation = data.to(device), targets.long().to(device), relation.to(device)

            output, K_embedding_batch = model(relation, data)
            loss = criterion(output, targets)

            total_loss += loss.item()  * len(data)

    return total_loss / len(data_loader)

In [94]:
# Function to calculate Cohen's kappa for two sets (confere <de Debora, C. and Luca, S., 2016. A class of k-modes algorithms for extracting knowledge structures from data. Behavior Research Methods.>)
def cohen_kappa(set1, set2):
    Q = set1.union(set2)  # possible knowledge states
    positive_agreements = len(set1.intersection(set2))
    negative_agreements = 2 * len(Q) - len(set1.union(set2))
    false_positives = len(set2.difference(set1))
    false_negatives = len(set1.difference(set2))
    observed_agreements = positive_agreements + negative_agreements
    total_pairs = len(Q) * 2  # Total possible pairs (agreements + disagreements)
    observed_disagreements = false_positives + false_negatives
    expected_agreement_chance = ((positive_agreements + false_negatives) * (positive_agreements + false_positives) +
                                 (false_positives + len(Q) - positive_agreements - false_positives) *
                                 (false_negatives + len(Q) - positive_agreements - false_negatives)) / (total_pairs**2)
    kappa = (observed_agreements / total_pairs - expected_agreement_chance) / (1 - expected_agreement_chance)
    return kappa

class CustomLRScheduler(_LRScheduler):
    def __init__(self, optimizer, init_lr, min_lr, adjust_factor, last_epoch=-1, verbose=False):
        self.init_lr = init_lr
        self.min_lr = min_lr
        self.last_lr = init_lr
        self.adjust_factor = adjust_factor
        self.current_phase_max_lr = init_lr
        self.is_increasing = False
        super(CustomLRScheduler, self).__init__(optimizer, last_epoch, verbose)

    def get_lr(self):
        if self.current_phase_max_lr <= self.min_lr:
            return [self.min_lr]

        if self.last_epoch == 0:
            return [self.init_lr]

        new_lr = 0
        if not self.is_increasing:
            new_lr = self.last_lr * self.adjust_factor
            if new_lr <= self.min_lr:
                new_lr = self.min_lr
                self.is_increasing = True
                self.min_lr = self.min_lr / 10
                self.current_phase_max_lr /= 10
        else:
            new_lr = self.last_lr / self.adjust_factor
            if new_lr >= self.current_phase_max_lr:
                self.is_increasing = False
                new_lr = self.current_phase_max_lr

        self.last_lr = new_lr
        return [new_lr]

def generate_sequence(model, input_matrix, start_token_id, eos_token_id, max_length=2**(N+1), precomputed_K_embedding=None):
    model.eval()
    generated_sequences = []
    K_embedding_batch = torch.Tensor()

    with torch.no_grad():
        sequence = [start_token_id]
        for _ in range(max_length):
            current_seq = torch.tensor([sequence], dtype=torch.long).long().to('cuda')
            output, K_embedding_batch = model(input_matrix, current_seq, precomputed_K_embedding) # Predict the next token
            next_token_id = output[-1].argmax(dim=-1).item()
            sequence.append(next_token_id)
            if next_token_id == eos_token_id: # Break if EOS token is generated
                break
        generated_sequences.append(sequence)

    return generated_sequences[0], K_embedding_batch

def latent_space_viz(model, VIZ_SET, full_dataset, device):
    indices = [0, len(dataset) - 1]
    max_index = max(enumerate(dataset), key=lambda e_seq: len(set(dataset.idx2state[token] for token in e_seq[1][1].tolist() if token != dataset.state2idx[PAD_TOKEN])))[0]
    indices += [max_index]
    indices += random.sample(range(1, len(dataset)-2), 2)
    VIZ_SET = Subset(full_dataset, indices) # minimal quasi-order for |Q|=2, for |Q|=5, biggest for |Q|=5, three random

    with torch.no_grad():
        K_embeddings = []
        target_list = []
        for data, targets, relation in VIZ_SET:
            input_matrix = relation.unsqueeze(0).long().to('cuda')
            eos_token_id = full_dataset.state2idx[EOS_TOKEN]
            start_token_id = full_dataset.state2idx['']
            generated_sequence, K_embedding_batch = generate_sequence(model, input_matrix, start_token_id=start_token_id, eos_token_id=eos_token_id)
            K_embeddings.append(K_embedding_batch)
            target_list.append(targets)

        cosine_similarity = torch.nn.CosineSimilarity(dim=0, eps=1e-6)
        print(f"Embedding inspection: {K_embeddings[0][0].shape}")
        print(f"Num viz sample: {len(K_embeddings)}")

        for idx, K_embed in enumerate(K_embeddings):
            print(f"    structure {alphabet[idx].upper()}: <<<{'|'.join([full_dataset.idx2state[token].ljust(6) for token in target_list[idx].tolist() if token != full_dataset.state2idx[PAD_TOKEN]])}>>> len {len([full_dataset.idx2state[token] for token in target_list[idx].tolist() if token != full_dataset.state2idx[PAD_TOKEN]])-1}")
            print(f"    mean: {K_embed.mean()} | std: {K_embed.std()} | min: {min(K_embed.flatten().tolist())} | max: {max(K_embed.flatten().tolist())} | magnitide: {torch.norm(K_embed)}")
            
        # calc similarity between K_embed and all other K_embeds from VIZ_SET
        comparison_prints = {}
        for i in range(len(K_embeddings)):
            for j in range(len(K_embeddings)):
                if i < j:
                    cosine = cosine_similarity(K_embeddings[i].flatten(), K_embeddings[j].flatten())
                    dot_similarity = torch.dot(K_embeddings[i].flatten(), K_embeddings[j].flatten())
                    euclidean = torch.dist(K_embeddings[i].flatten(), K_embeddings[j].flatten(), p=2)
                    t1 = set([state for state in target_list[i].tolist() if state != full_dataset.state2idx[PAD_TOKEN]])
                    t2 = set([state for state in target_list[j].tolist() if state != full_dataset.state2idx[PAD_TOKEN]])
                    kappa = cohen_kappa(t1, t2)
                    jaccard_idx = len(t1.intersection(t2)) / len(t1.union(t2))
                    comparison_prints[kappa] =  f"    - similarity {alphabet[i].upper()}-{alphabet[j].upper()} ->  cosine: {str(round(cosine.item(), 3)).ljust(7)} | dot: {str(round(dot_similarity.item(), 3)).ljust(7)} | euclidean: {str(round(euclidean.item(), 3)).ljust(7)} || cohen's k: {str(round(kappa, 3)).ljust(7)} | jaccard: {str(round(jaccard_idx, 3)).ljust(7)}"
        for k in sorted(comparison_prints.keys()):
            print(comparison_prints[k])

def performance(model, dataset, full_dataset, device):
    model.eval()
    correct = 0
    total = 0
    norm_len_diffs = []

    subset = random.sample(list(dataset), 100) # Select a random subset of 100 samples from the dataset
    print_n = 5
    counter = 0

    with torch.no_grad():
        for data, targets, relation in tqdm(subset):
            data, targets, relation = data.to(device), targets.long().to(device), relation.to(device)

            input_matrix = relation.unsqueeze(0).long().to('cuda')
            eos_token_id = full_dataset.state2idx[EOS_TOKEN]
            start_token_id = full_dataset.state2idx['']
            generated_sequence, K_embedding_batched = generate_sequence(model, input_matrix, start_token_id=start_token_id, eos_token_id=eos_token_id)

            check = True
            for i in range(len(generated_sequence)-1):
                if generated_sequence[i+1] != targets[i]:
                    check = False
                    break
            if check:
                correct += 1
            else:
                # print target and generated sequence
                if counter < print_n:
                    counter += 1
                    target_tokens = [full_dataset.idx2state[t.item()] for t in targets]
                    target_str = str(counter) + ") Target: |" + "|".join([token.ljust(6) for token in target_tokens])
                    print("\n", target_str)
                    seq_tokens = [full_dataset.idx2state[idx] for idx in generated_sequence]
                    seq_str = ">>>         |" + "|".join([token.ljust(6) for token in seq_tokens[1:]])
                    print(seq_str)
            total += 1
            
            embedding_norms = K_embedding_batched.norm(dim=1)
            targets = targets.unsqueeze(0)
            seq_lengths = ((targets != full_dataset.state2idx[PAD_TOKEN]) & (targets != full_dataset.state2idx[EOS_TOKEN])).sum(dim=1).float()
            diff = embedding_norms - seq_lengths
            norm_len_diffs += [abs(diff.item())]
            
    print(f"Accuracy: {correct / total}")
    print(f"Mean length difference: {sum(norm_len_diffs) / len(norm_len_diffs)}")
    return correct / total

### Test functions

In [ ]:
### Test training and evaluation:

# # Model parameters
# hidden_dim = N_HIDDEN
# n_layers = 1
# n_heads = 1
# dropout = 0.2
# # Initialize model
# input_dim = N**2
# n_tokens = dataset.vocab_size
# model = SimpleDecoderModel(input_dim, hidden_dim, n_tokens, n_heads, n_layers, dropout)
# lr = 0.005
# optimizer = optim.Adam(model.parameters(), lr=lr)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)
# clip_norm  = 0.5
# criterion1 = custom_CE_loss
# criterion2 = nn.MSELoss()
# length_wheight = 0.5
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# optimizer.zero_grad()
# dataset = Sequences(domain_sizes=[3])
# train_loader = DataLoader(dataset, batch_size=BATCH_SIZE)

# # Train model
# model = model.to(device)
# print(train(model, train_loader, optimizer, criterion1, criterion2, scheduler, clip_norm, dataset, length_wheight, device)) # to do: pass log_interval rate
# print(evaluate(model, train_loader, criterion1, device))

### Test kappa calculation:
# # Define the sets based on binary representation to knowledge structures
# structures = {
#     'a': {'0000', '1000', '0100', '0010', '0001', '1100', '1010', '1001', '0110', '0101', '0011', '1110', '1101', '1011', '0111', '1111'},
#     'b': {'0000', '1000', '1100', '1110', '1111'},
#     'c': {'0000', '0100', '0010', '0110', '0101', '0111', '1111'},
#     'd': {'0000', '1000', '0010', '1010', '0110', '0011', '1110', '1011', '0111', '1111'}
# }
# # Calculate kappa for all pairs of structures
# kappa_values = {}
# for key1, value1 in structures.items():
#     for key2, value2 in structures.items():
#         if key1 < key2:  # To ensure each pair is calculated only once
#             kappa_values[f'{key1} vs {key2}'] = cohen_kappa(value1, value2)
# print(f"Test Kappa: {kappa_values}")


### Test data subset for visualization
# indices = [0, len(dataset) - 1, random.randint(1, len(dataset) - 2)]
# print(f"Test visualization set: Indices {indices}")
# VIZ_SET = Subset(dataset, indices)
# # print(dataset[0][0])
# for i in range(len(VIZ_SET)):
#     In, T, S = next(iter(VIZ_SET))
#     print(T)
#     print(dataset[indices[0]][2])


### Test CustomLRScheduler
# LR = 0.01
# ADJUST_FACTOR = 0.8
# MIN_LR = 0.0001
# optimizer = torch.optim.SGD([torch.nn.Parameter(torch.randn(2, 2))], lr=LR)
# scheduler = CustomLRScheduler(optimizer, init_lr=LR, min_lr=MIN_LR, adjust_factor=ADJUST_FACTOR) # behaves like normal stepLR for minLR = LR/10, otherwise a linear cosine annealing scheduler...

# for epoch in range(1, 300):
#     # print(f"loop - epoch {epoch}")
#     lr = scheduler.get_lr()[0]
#     formatted_lr = f"{lr:.10f}".rstrip('0').rstrip('.')
#     print(f"Epoch {epoch}, LR: {formatted_lr}")
#     scheduler.step()

### Run

In [65]:
# Dataset
domain_sizes = [5] # list(range(2,N+1))
dataset = Sequences(domain_sizes=domain_sizes)
complete_loader = DataLoader(dataset, batch_size=BATCH_SIZE)

# Define the proportions
train_size = int(0.7 * len(dataset))  # 70% for training
val_size = int(0.15 * len(dataset))   # 15% for validation
test_size = len(dataset) - train_size - val_size  # 15% for testing
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

indices =  [0,1,2,3,4] ### will be changed every train epoch anyways..... to do...
VIZ_SET = Subset(dataset, indices)

# Model parameters and initialization
hidden_dim = N_HIDDEN
n_layers = 1
n_heads = 1
dropout = 0.3
input_dim = N**2
n_tokens = dataset.vocab_size
model = SimpleDecoderModel(input_dim, hidden_dim, n_tokens, n_heads, n_layers, dropout)

LR = 0.001 # 0.001
optimizer = optim.Adam(model.parameters(), lr=LR) # replaced for L2.....
# decoder_params = list(model.output_proj.parameters()) + list(model.output_proj.parameters())
# decoder_params_ids = {id(p) for p in decoder_params}
# other_params = [p for p in model.parameters() if id(p) not in decoder_params_ids]
# weight_decay_for_decoder = 0.01  
# param_groups = [
#     {'params': decoder_params, 'weight_decay': weight_decay_for_decoder},
#     {'params': other_params, 'weight_decay': 0.0},]  # No L2 regularization for these parameters
# optimizer = optim.Adam(param_groups, lr=LR)


ADJUST_FACTOR = 0.8
MIN_LR = 0.0001  # # # 0.01 
scheduler = CustomLRScheduler(optimizer, init_lr=LR, min_lr=MIN_LR, adjust_factor=ADJUST_FACTOR)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.90)
clip_norm  = 0.8
criterion = custom_CE_loss

MARGIN_SIZE = 1.0 # size (length) of the cmse tolerance margin 
FACTOR = 0.2 
# def modified_mse(y_pred, y_true): # introduces a tolerance margin to the calculation in order not to "discretize" the latent/embedding space
#     diff = y_pred - y_true
#     diff = torch.sign(diff) * torch.clamp(torch.abs(diff) - MARGIN_SIZE/2, min=0)
#     return (diff ** 2).mean()

def modified_mse(y_pred, y_true):
    diff = y_pred - y_true
    abs_diff = torch.abs(diff)

    scale_combined = torch.zeros_like(y_pred)
    if MARGIN_SIZE/2 >= 1:
        scale1 = torch.where(abs_diff <= MARGIN_SIZE / 2, abs_diff * MARGIN_SIZE/2, torch.ones_like(abs_diff))
        mse_value_at_abs_diff = scale1.max()
        scale2 = torch.where(abs_diff <= MARGIN_SIZE / 2, scale1 / mse_value_at_abs_diff, scale1)
        scale_combined = torch.where(abs_diff <= MARGIN_SIZE/2, scale2**FACTOR, scale1)
    else:
        scale1 = torch.where(abs_diff <= MARGIN_SIZE / 2, abs_diff * MARGIN_SIZE/2, torch.ones_like(abs_diff))
        mse_value_at_abs_diff = (MARGIN_SIZE/2) ** 2
        scale2 = torch.where(abs_diff <= MARGIN_SIZE / 2, scale1 / mse_value_at_abs_diff / MARGIN_SIZE, scale1)
        scale_combined = torch.where(abs_diff <= MARGIN_SIZE/2, scale2**FACTOR, scale1)

    scaled_diff = scale_combined * abs_diff
    modified_mse = (scaled_diff ** 2).mean()
    return modified_mse

# criterion2 = nn.HuberLoss()
# criterion2 = nn.MSELoss()  
criterion2 = modified_mse 
length_wheight = 0.7


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
optimizer.zero_grad()
model = model.to(device)
save_dir = 'data/DecoderTransformer-v01_checkpoints'

def training(epochs):
    ### initialize training

    # Training metrics
    train_losses = []
    val_losses = []
    acc_values = []

    total_epoch = 0

    # Loop 1
    for _ in range(epochs[0]):
        latent_space_viz(model, VIZ_SET, dataset, device)

        # Loop 2
        for e in range(epochs[1]):
            train_losses += [train(model, train_loader, optimizer, criterion, criterion2, scheduler, clip_norm, dataset, length_wheight, device)]
            val_losses += [evaluate(model, val_loader, criterion, device)]
            total_epoch += 1
            if total_epoch % 1 == 0:
                print(f"Epoch {total_epoch}/{epochs[0]*epochs[1]}: Train Losses: {train_losses[-1][-1]:.6f} (CE{train_losses[-1][0]:.6f}) (L{train_losses[-1][1]:.6f}) | Validation Loss: {val_losses[-1]:.6f} | LR: {scheduler.get_last_lr()[0]:.11f}")
            scheduler.step()

        # Evaluate performance on complete dataset sample
        acc_values += [performance(model, test_dataset, dataset, device)]
        print()

    return train_losses, val_losses

Initialized properly.


100%|██████████| 4231/4231 [00:00<00:00, 162756.00it/s]


In [66]:
train_losses, val_losses = training([100,3])

Embedding inspection: torch.Size([1024])
Num viz sample: 5
    structure A: <<<a     |b     |c     |d     |e     |ab    |ac    |ad    |ae    |bc    |bd    |be    |cd    |ce    |de    |abc   |abd   |abe   |acd   |ace   |ade   |bcd   |bce   |bde   |cde   |abcd  |abce  |abde  |acde  |bcde  |abcde |<eos> >>> len 31
    mean: -0.018015054985880852 | std: 0.29610583186149597 | min: -0.9258952140808105 | max: 0.7915346622467041 | magnitide: 9.488287925720215
    structure B: <<<a     |ab    |abc   |abcd  |abcde |<eos> >>> len 5
    mean: -0.02750527672469616 | std: 0.4580502212047577 | min: -1.4416553974151611 | max: 1.3352752923965454 | magnitide: 14.676864624023438
    structure C: <<<a     |b     |c     |d     |e     |ab    |ac    |ad    |ae    |bc    |bd    |be    |cd    |ce    |de    |abc   |abd   |abe   |acd   |ace   |ade   |bcd   |bce   |bde   |cde   |abcd  |abce  |abde  |acde  |bcde  |abcde |<eos> >>> len 31
    mean: -0.018015054985880852 | std: 0.29610583186149597 | min: -0.92589521

  0%|          | 0/100 [00:00<?, ?it/s]


 1) Target: |c     |ac    |ace   |abce  |acde  |abcde |<eos> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> 
>>>         |c     |ac    |ce    |ace   |abce  |acde  |abcde |<eos> 


  2%|▏         | 2/100 [00:00<00:05, 17.24it/s]


 2) Target: |a     |b     |d     |ab    |ad    |bd    |cd    |abd   |acd   |ade   |bcd   |abcd  |abde  |acde  |abcde |<eos> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> |<pad> 
>>>         |a     |b     |d     |ad    |bd    |cd    |abd   |acd   |ade   |bcd   |abcd  |abde  |acde  |abcde |<eos> 


...

100%|██████████| 100/100 [00:09<00:00, 10.31it/s]


Accuracy: 1.0
Mean length difference: 0.5563496828079224

Embedding inspection: torch.Size([1024])
Num viz sample: 5
    structure A: <<<a     |b     |c     |d     |e     |ab    |ac    |ad    |ae    |bc    |bd    |be    |cd    |ce    |de    |abc   |abd   |abe   |acd   |ace   |ade   |bcd   |bce   |bde   |cde   |abcd  |abce  |abde  |acde  |bcde  |abcde |<eos> >>> len 31
    mean: -0.008897591382265091 | std: 0.6478283405303955 | min: -1.4036355018615723 | max: 1.3000752925872803 | magnitide: 20.722339630126953
    structure B: <<<a     |ab    |abc   |abcd  |abcde |<eos> >>> len 5
    mean: -0.0021022444125264883 | std: 0.18586848676204681 | min: -0.5421057343482971 | max: 0.6949805617332458 | magnitide: 5.945267677307129
    structure C: <<<a     |b     |c     |d     |e     |ab    |ac    |ad    |ae    |bc    |bd    |be    |cd    |ce    |de    |abc   |abd   |abe   |acd   |ace   |ade   |bcd   |bce   |bde   |cde   |abcd  |abce  |abde  |acde  |bcde  |abcde |<eos> >>> len 31
    mean: -0.0088

100%|██████████| 100/100 [00:09<00:00, 10.26it/s]


Accuracy: 1.0
Mean length difference: 0.37591817378997805

Embedding inspection: torch.Size([1024])
Num viz sample: 5
    structure A: <<<a     |b     |c     |d     |e     |ab    |ac    |ad    |ae    |bc    |bd    |be    |cd    |ce    |de    |abc   |abd   |abe   |acd   |ace   |ade   |bcd   |bce   |bde   |cde   |abcd  |abce  |abde  |acde  |bcde  |abcde |<eos> >>> len 31
    mean: -0.008888340555131435 | std: 0.6478400826454163 | min: -1.4035230875015259 | max: 1.299922227859497 | magnitide: 20.72270965576172
    structure B: <<<a     |ab    |abc   |abcd  |abcde |<eos> >>> len 5
    mean: -0.0020924736745655537 | std: 0.1860487014055252 | min: -0.5436136722564697 | max: 0.6949222087860107 | magnitide: 5.9510273933410645
    structure C: <<<a     |b     |c     |d     |e     |ab    |ac    |ad    |ae    |bc    |bd    |be    |cd    |ce    |de    |abc   |abd   |abe   |acd   |ace   |ade   |bcd   |bce   |bde   |cde   |abcd  |abce  |abde  |acde  |bcde  |abcde |<eos> >>> len 31
    mean: -0.00888

KeyboardInterrupt: 

In [ ]:
# save_dir = 'data/DecoderTransformer-v01_checkpoints'
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)
# # save torch model
# model_save_name = 'DecoderTransformer-v01.pt'
# save_path = f"{save_dir}/{model_save_name}"
# torch.save(model.state_dict(), save_path)

In [ ]:
# reload model

### Latent space Interpolation

In [103]:
indices = [0,len(dataset)-1]
indices += random.sample(range(1, len(dataset)-2), 3)
VIZ_SET = Subset(dataset, indices)
eos_token_id = dataset.state2idx[EOS_TOKEN]
start_token_id = dataset.state2idx['']
model.eval()

K_embeddings = []
output_seqs = []
for _, _, relation in VIZ_SET:
    input_matrix = relation.unsqueeze(0).long().to('cuda')
    
    output_seq, K_embedding_batch = generate_sequence(model, input_matrix, start_token_id=start_token_id, eos_token_id=eos_token_id)
    K_embeddings.append(K_embedding_batch)
    output_seqs.append(output_seq)
    
    # seq_tokens = [dataset.idx2state[idx] for idx in output_seq]
    # seq_str = "Start sequences: |" + "|".join([token.ljust(6) for token in seq_tokens[1:]])
    # print(seq_str)

# interpolate between two K_embeddings
intermediate_embeddings = []
N_interpol = 100
for i in list(range(N_interpol+1)):
    alpha = i / N_interpol
    intermediate_embedding = alpha * K_embeddings[0] + (1 - alpha) * K_embeddings[1]
    intermediate_embeddings.append(intermediate_embedding)

# run model on intermediate embeddings (with start token, without input projection)
intermediate_seqs = []
# set all elements of tensor to zero
input_matrix = input_matrix * 0
for i in list(range(N_interpol+1)):
    # place embeddings into memory cell and run forward pass without projection layer
    precomputed_K_embedding = intermediate_embeddings[i].to('cuda')
    output_seqs, _ = generate_sequence(model, input_matrix, start_token_id=start_token_id, eos_token_id=eos_token_id, precomputed_K_embedding=precomputed_K_embedding)
    intermediate_seqs.append(output_seqs)

for i, seq in enumerate(intermediate_seqs): 
    seq_tokens = [dataset.idx2state[idx] for idx in seq]
    seq_str = f" >>> Sequence {i}: |" + "|".join([token.ljust(6) for token in seq_tokens[1:]])
    print(seq_str)


 >>> Sequence 0: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 1: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 2: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 3: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 4: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 5: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 6: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 7: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 8: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 9: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 10: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 11: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 12: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 13: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 14: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 15: |a     |ab    |abc   |abcd  |abcde |<eos> 
 >>> Sequence 16: |a     |ab    |a